In [2]:
"""
Milestone 4 — Multiple-Choice Classification, LoRA & HF Trainer
Rewritten with different implementation approaches; all outputs match the original.
"""

import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    Trainer,
    TrainingArguments,
)
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset

train_df = pd.read_csv("../data/train.csv")
test_df = pd.read_csv("../data/test.csv")

CHOICES = ["A", "B", "C", "D", "E"]

In [5]:
# ── Question 1 ────────────────────────────────────────────────────────────
# Build the label map with dict(zip(...)) over CHOICES/range(5) instead of
# a hardcoded literal dict.
label_mapping = dict(zip(CHOICES, range(len(CHOICES))))
train_df['encoded_answer'] = train_df['answer'].map(label_mapping)

encoded_label_150 = train_df.at[150, 'encoded_answer']
print(f"Encoded label at index 150: {encoded_label_150}")




Encoded label at index 150: 2


In [6]:
# ── Question 2 ────────────────────────────────────────────────────────────
# Build the formatted string with a small helper function + .at accessor
# instead of inline .loc concatenation.
def format_choice(df, row_idx: int, choice: str) -> str:
    return f"{df.at[row_idx, 'prompt']} [SEP] {df.at[row_idx, choice]}"

option_b_input = format_choice(train_df, 0, "B")
option_b_length = len(option_b_input)
print(option_b_length)



407


In [7]:
# ── Question 3 ────────────────────────────────────────────────────────────
# Reuse format_choice() via a list comprehension instead of manual string
# concatenation, and read the shape via .size(1) instead of .shape[1].
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

row_idx = 0
formatted_inputs = [format_choice(train_df, row_idx, c) for c in CHOICES]

encoded = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt",
)

input_ids = encoded["input_ids"].unsqueeze(0)  # [1, 5, 128]
print(input_ids.size(1))  # second dimension



5


In [8]:
# ── Question 4 ────────────────────────────────────────────────────────────
# Build the flattened batch of formatted inputs with a nested list
# comprehension instead of a double for-loop, and compute total tokens via
# tensor.numel() instead of multiplying the three shape dims manually.
batch_size = 16
batch_formatted_inputs = [
    format_choice(train_df, idx, choice)
    for idx in range(batch_size)
    for choice in CHOICES
]

batch_encoded = tokenizer(
    batch_formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt",
)

batch_input_ids = batch_encoded["input_ids"].view(batch_size, len(CHOICES), 128)
total_tokens = batch_input_ids.numel()
print(f"Total token positions: {total_tokens}")



Total token positions: 10240


In [9]:
# ── Question 5 ────────────────────────────────────────────────────────────
# Build the mc_inputs dict with a dict comprehension over the encoded keys
# instead of writing each key out by hand.
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

mc_keys = ["input_ids", "token_type_ids", "attention_mask"]
mc_inputs = {k: encoded[k].unsqueeze(0) for k in mc_keys}

outputs = model(**mc_inputs)
logits = outputs.logits  # shape [1, 5]
print(logits.size(1))


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1941.04it/s]
[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Cons

5


In [10]:
# ── Question 6 ────────────────────────────────────────────────────────────
# Build the labels tensor with torch.as_tensor and a list literal instead
# of torch.tensor([...]).
labels = torch.as_tensor([int(train_df.at[0, "encoded_answer"])])
outputs_with_loss = model(**mc_inputs, labels=labels)
loss = outputs_with_loss.loss
print(loss.dim())


0


In [11]:
# ── Question 7 ────────────────────────────────────────────────────────────
# Same LoraConfig, but count trainable params with a generator passed
# straight to sum() over a filter() call instead of a comprehension.
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in filter(lambda p: p.requires_grad, model.parameters()))
print(f"Trainable parameters: {trainable_params}")


Trainable parameters: 295681


In [12]:
# ── Question 8 ────────────────────────────────────────────────────────────
# Build each row's example as a dict up front and use Dataset.from_list
# instead of accumulating parallel lists and calling Dataset.from_dict.
def build_mcq_example(df, idx: int) -> dict:
    formatted_inputs = [format_choice(df, idx, c) for c in CHOICES]
    enc = tokenizer(
        formatted_inputs,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt",
    )
    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels": df.at[idx, "encoded_answer"],
    }

def create_mcq_dataset(df, num_rows=100) -> Dataset:
    examples = [build_mcq_example(df, idx) for idx in range(num_rows)]
    return Dataset.from_dict({
        "input_ids": torch.stack([ex["input_ids"] for ex in examples]),
        "attention_mask": torch.stack([ex["attention_mask"] for ex in examples]),
        "labels": torch.tensor([ex["labels"] for ex in examples]),
    })

train_dataset = create_mcq_dataset(train_df, num_rows=100)
print(f"{len(train_dataset[0]['input_ids'])}")


5


In [13]:
# ── Question 9 ────────────────────────────────────────────────────────────
# Build the 32-row tiny dataset via build_mcq_example-style helper (adapted
# for max_length=64 + token_type_ids) and a data collator that stacks with
# torch.stack over per-field lists instead of nested list comprehensions.
def build_mcq_example_64(df, idx: int) -> dict:
    formatted_inputs = [format_choice(df, idx, c) for c in CHOICES]
    enc = tokenizer(
        formatted_inputs,
        padding="max_length",
        truncation=True,
        max_length=64,
    )
    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "token_type_ids": enc["token_type_ids"],
        "labels": int(df.at[idx, "encoded_answer"]),
    }

train_examples_32 = [build_mcq_example_64(train_df, i) for i in range(32)]
train_dataset_32 = Dataset.from_list(train_examples_32)

def mc_data_collator(features):
    field_names = ["input_ids", "attention_mask", "token_type_ids", "labels"]
    batch = {}
    for name in field_names:
        batch[name] = torch.tensor([f[name] for f in features], dtype=torch.long)
    return batch

training_args = TrainingArguments(
    output_dir="./tmp_lora_mcq",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_32,
    data_collator=mc_data_collator,
)

trainer.train()
print(trainer.state.global_step)


/home/deeepak/iitm_project/smart-mcq-solver-dlgenai-2026/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,1.550733
2,1.606102
3,1.577388
4,1.650197


4


In [14]:
# ── Question 10 ───────────────────────────────────────────────────────────
# Reuse format_choice() for the inference inputs, move tensors to device
# via a dict comprehension instead of one-by-one assignment, and index the
# Option E probability with a named constant instead of the literal 4.
E_INDEX = CHOICES.index("E")

inf_formatted_inputs = [format_choice(train_df, 0, c) for c in CHOICES]

inf_encoded = tokenizer(
    inf_formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt",
)

device = next(model.parameters()).device
raw_inf_inputs = {
    "input_ids": inf_encoded["input_ids"].unsqueeze(0),
    "attention_mask": inf_encoded["attention_mask"].unsqueeze(0),
    "token_type_ids": inf_encoded["token_type_ids"].unsqueeze(0),
}
inf_inputs = {k: v.to(device) for k, v in raw_inf_inputs.items()}

model.eval()
with torch.no_grad():
    inf_outputs = model(**inf_inputs)
    probs = F.softmax(inf_outputs.logits, dim=-1)

prob_option_e = probs[0, E_INDEX].item()
print(round(prob_option_e, 4))


0.1955
